# שיעור 11 - פרוטוקול הקשר של המודל (MCP)

ה-**פרוטוקול הקשר של המודל (MCP)** הוא תקן פתוח שמאפשר לסוכנים לגלות ולהשתמש בצורה דינמית בכלים, משאבים ומקורות נתונים בזמן ריצה. במקום לקודד כלים ישירות בתוך סוכן, MCP מאפשר לסוכנים להתחבר לשרתים חיצוניים שמחשפים יכולות לפי דרישה.

במשיעור זה תלמדו:
- מהו MCP ולמה הוא חשוב עבור מערכות סוכנים
- כיצד פועלת ארכיטקטורת לקוח-שרת של MCP
- כיצד לבנות סוכנים שמשתמשים בגילוי כלים בסגנון MCP


## הגדרה

**דרישות מוקדמות:**
- פרויקט Azure AI Foundry עם מודל פרוס
- הרץ `az login` לאימות


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## מהו פרוטוקול הקשר של המודל (MCP)?

MCP מגדיר דרך סטנדרטית לסוכני בינה מלאכותית לגלות ולהתממשק עם כלים חיצוניים ומקורות נתונים:

- **שרת MCP**: חושף כלים, משאבים ופרומפטים באמצעות פרוטוקול סטנדרטי
- **לקוח MCP**: סביבת הריצה של הסוכן שמתחברת לשרתים ומגלה את היכולות הזמינות
- **גילוי דינמי**: לסוכנים אין צורך בכלים מקודדים מראש — הם מגלים מה זמין בזמן ריצה

זוהי יכולת חזקה לבניית מערכות סוכנים הניתנות להרחבה, שבהן ניתן להוסיף יכולות חדשות מבלי לשנות את קוד הסוכן.


## איך MCP עובד

```
┌─────────────┐     discover      ┌─────────────────┐
│  MCP Client  │ ──────────────► │   MCP Server     │
│  (Agent)     │                  │  (Tool Provider) │
│              │ ◄────────────── │                   │
│              │   tool results   │  • list_tools()  │
│              │                  │  • call_tool()   │
└─────────────┘                  │  • resources     │
                                  └─────────────────┘
```

1. הסוכן (MCP client) מתחבר ל-MCP server
2. השרת מגיב עם רשימה של הכלים הזמינים והסכמות שלהם
3. הסוכן יכול אז להפעיל כל כלי שהתגלה במהלך תהליך החשיבה שלו
4. התוצאות זורמות בחזרה באמצעות אותו פרוטוקול


## הדמיית גילוי כלים של MCP

מכיוון ששרת MCP אמיתי דורש תהליך שרת פועל, נדגים את הדפוס באמצעות פונקציות `@tool` שמדמות את מה ששירות אירוח המחובר ל-MCP יספק.

בסביבת ייצור, כלים אלה יתגלו באופן דינמי משרת MCP במקום להיות מוגדרים באופן מקומי.


In [ ]:
@tool(approval_mode="never_require")
def search_accommodations(
    location: Annotated[str, "The city to search for accommodations"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"] = 2
) -> str:
    """Search for accommodations (simulating an MCP-connected Airbnb tool).
    In production, this would be discovered via MCP from an accommodation service."""
    listings = {
        "Tokyo": [
            {"name": "Shinjuku Modern Apartment", "price": 120, "rating": 4.8},
            {"name": "Traditional Ryokan in Asakusa", "price": 200, "rating": 4.9},
            {"name": "Shibuya Studio", "price": 85, "rating": 4.5},
        ],
        "Paris": [
            {"name": "Le Marais Charming Flat", "price": 150, "rating": 4.7},
            {"name": "Montmartre Artist Loft", "price": 110, "rating": 4.6},
        ],
        "Barcelona": [
            {"name": "Gothic Quarter Penthouse", "price": 130, "rating": 4.8},
            {"name": "Barceloneta Beach Flat", "price": 95, "rating": 4.4},
        ],
    }
    results = listings.get(location, [])
    if not results:
        return f"No accommodations found in {location}"
    output = f"Accommodations in {location} ({check_in} to {check_out}, {guests} guests):\n"
    for listing in results:
        output += f"  - {listing['name']}: ${listing['price']}/night (★{listing['rating']})\n"
    return output


@tool(approval_mode="never_require")
def get_local_experiences(
    location: Annotated[str, "The city to find experiences in"],
    interest: Annotated[str, "Type of experience (food, culture, adventure, etc.)"] = "all"
) -> str:
    """Get local experiences and activities (simulating an MCP-connected tourism tool)."""
    experiences = {
        "Tokyo": {
            "food": ["Tsukiji Market Tour ($45)", "Ramen Making Class ($60)", "Sake Tasting ($35)"],
            "culture": ["Tea Ceremony ($50)", "Samurai Museum ($15)", "Sumo Tournament ($80)"],
            "adventure": ["Mt. Fuji Day Trip ($120)", "Go-kart City Tour ($80)"],
        },
        "Paris": {
            "food": ["Wine & Cheese Tasting ($55)", "Cooking Class ($90)", "Market Tour ($40)"],
            "culture": ["Louvre Guided Tour ($35)", "Montmartre Art Walk ($25)"],
        },
    }
    city_exp = experiences.get(location, {})
    if not city_exp:
        return f"No experiences found in {location}"
    if interest != "all" and interest in city_exp:
        items = city_exp[interest]
        return f"{interest.title()} experiences in {location}:\n" + "\n".join(f"  - {e}" for e in items)
    output = f"All experiences in {location}:\n"
    for cat, items in city_exp.items():
        output += f"\n  {cat.title()}:\n"
        for item in items:
            output += f"    - {item}\n"
    return output

## בניית סוכן עם כלים בסגנון MCP


In [ ]:
agent = await provider.create_agent(
    tools=[search_accommodations, get_local_experiences],
    name="AccommodationAgent",
    instructions="""You are an accommodation and travel experiences specialist powered by MCP-connected services.

Help travelers find the perfect place to stay and things to do. When searching:
1. Use the search_accommodations tool to find listings
2. Use the get_local_experiences tool to suggest activities
3. Compare options and make personalized recommendations
4. Consider the traveler's budget, interests, and travel style""",
)

response = await agent.run(
    "I'm visiting Tokyo for 5 nights in April with my partner. We love traditional Japanese culture and food. "
    "Find us a place to stay and suggest some experiences.",
    )
print(response)

## MCP בסביבת ייצור

בסביבת ייצור, MCP מאפשרת דפוסים רבי-עוצמה:

- **איתור כלים דינמי**: סוכנים מתחברים לשרתי MCP ומגלים כלים בזמן ריצה
- **ארכיטקטורה מנותקת**: ספקי כלים יכולים לעדכן באופן עצמאי מהסוכן
- **שיתוף בין-ארגוני**: צוותים יכולים לחשוף יכולות דרך שרתי MCP שכל סוכן יכול להשתמש בהן
- **תמיכה ב‑Microsoft Agent Framework**: ל‑MAF יש תמיכה מובנית בלקוח MCP דרך האינטגרציה `mcp`

כדי להשתמש בשרת MCP אמיתי עם MAF, תתחברו דרך `hosted_mcp_tool()` או דרך אינטגרציית לקוח ה‑MCP.

**למידע נוסף:**
- [מפרט MCP](https://modelcontextprotocol.io/)
- [תמיכה של Microsoft Agent Framework ב‑MCP](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/mcp)


## סיכום

בשיעור זה למדת:
- **MCP** הוא תקן פתוח לגילוי דינמי של כלים בין סוכנים וספקי כלים
- **ארכיטקטורת לקוח-שרת** מאפשרת לסוכנים לגלות יכולות בזמן ריצה
- MCP מאפשר **מערכות סוכנים ניתקות ומותאמות להרחבה** שבהן ניתן להוסיף כלים ללא שינויי קוד
- Microsoft Agent Framework מספק **תמיכה מובנית ב‑MCP** לשימוש בייצור


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
כתב ויתור:
מסמך זה תורגם באמצעות שירות תרגום מבוסס בינה מלאכותית Co-op Translator (https://github.com/Azure/co-op-translator). למרות שאנו שואפים לדיוק, יש לקחת בחשבון שתרגומים אוטומטיים עלולים להכיל שגיאות או אי-דיוקים. יש להסתמך על המסמך המקורי בשפת המקור כמקור הסמכות. עבור מידע קריטי מומלץ לפנות לתרגום מקצועי שמבוצע על ידי מתרגם אנושי. איננו אחראים לכל אי-הבנות או לפרשנויות שגויות הנובעות מהשימוש בתרגום זה.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
